# 40 · Analysis: which jobs is NYC adding?

**Rules:** written before these results. Reads: `fct_job_projection` (2022-32, D1/D15 fill), `fct_job_employment_nyc_est` (D8/D18 estimate), `fct_job_employment_wage_annual` (OEWS NY metro, published), `fct_job_wage_nyc` (NYS DOL entry wage), `dim_job`, and NYS DOL's Employment Prospects rating from staging as an external check. Changes nothing.

**Questions this notebook answers**
1. How big is the degree-reachable part of NYC's job market, and how has it moved 2019 → 2025?
2. The S10 lists: the degree-reachable jobs with the most openings (projected) and the fastest actual growth since 2019, with starting pay beside each.
3. Do the two independent series (NYC estimate, metro published) and NYS DOL's own rating agree with those lists? Where they don't, why?

**Rules applied (plan §3a–3b):** universe = `is_degree_entry` detailed jobs; All Other codes in totals, out of the top lists; growth floor = 1,000 estimated NYC jobs in both 2019 and 2025; 2019 split codes flagged.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from explore import q

pd.set_option("display.max_columns", 40, "display.max_colwidth", 58, "display.width", 240,
              "display.float_format", "{:,.2f}".format)


def md(s: str):
    display(Markdown(s))


def show(df: pd.DataFrame, title: str | None = None, n: int | None = None):
    if title:
        md(f"**{title}**")
    display(df if n is None else df.head(n))

## 1. The job universe

`dim_job` carries BLS's typical entry education. A job is degree-reachable when that is an associate's or a bachelor's (`is_degree_entry`, S5). Everything else is counted but not listed.

In [2]:
jobs_all = q("""
select soc_code, soc_title, soc_level, major_code, entry_education, entry_education_rank, is_degree_entry, is_all_other,
       in_tech, in_health, in_business, in_education, ai_band
from dim_job
""")
majors = q("select soc_code as major_code, soc_title as major_title from dim_job where soc_level = 'major'").set_index("major_code")["major_title"]
jobs_all["major_title"] = jobs_all.major_code.map(majors)
jobs = jobs_all[jobs_all.soc_level == "detailed"].copy()

uni = jobs[jobs.is_degree_entry]
n_assoc = int((uni.entry_education == "Associate's degree").sum())
n_bach = int((uni.entry_education == "Bachelor's degree").sum())
print(f"detailed jobs in dim_job: {len(jobs):,} | degree-reachable: {len(uni):,} "
      f"(associate's {n_assoc}, bachelor's {n_bach}) | All Other among them: {int(uni.is_all_other.sum())}")
show(jobs.groupby("entry_education", dropna=False).size().rename("jobs").to_frame()
        .join(jobs.groupby("entry_education", dropna=False).is_degree_entry.first())
        .sort_values("jobs", ascending=False), "Detailed jobs by BLS entry education")

detailed jobs in dim_job: 867 | degree-reachable: 221 (associate's 48, bachelor's 173) | All Other among them: 23


**Detailed jobs by BLS entry education**

,jobs,is_degree_entry
entry_education,,
High school diploma or equivalent,320,False
Bachelor's degree,173,True
No formal educational credential,109,False
Doctoral or professional degree,73,False
Postsecondary nondegree award,51,False
Associate's degree,48,True
NaN,48,False
Master's degree,39,False
"Some college, no degree",6,False


## 2. NYC employment 2019 → 2025, degree-reachable vs the rest

`fct_job_employment_nyc_est` is the D8 estimate (QCEW NYC × same-year NY staffing pattern, D18 government filing). Its yearly sum equals the QCEW NYC total by construction, so the split between degree-reachable and other jobs is the thing to read, not the total. The OEWS metro series (published, twice NYC's size) sits beside it as the independent check.

In [3]:
est = q("""
select e.year, e.soc_code, e.employment_est, e.metro_employment, e.n_sectors_contributing, e.share_from_government_pattern,
       e.nyc_wage_table_employment
from fct_job_employment_nyc_est e
""")
print(f"fct_job_employment_nyc_est rows read: {len(est):,} ({est.soc_code.nunique()} jobs × {est.year.nunique()} years)")
est = est.merge(jobs_all[["soc_code", "soc_level", "is_degree_entry", "is_all_other", "entry_education", "major_code", "major_title"]], on="soc_code", how="left")
assert est.is_degree_entry.notna().all(), "every estimated job must be in dim_job"
nd = est[(est.year == 2025) & (est.soc_level != "detailed")]
print(f"estimated codes that are broad/aggregate rows in dim_job (OEWS research publishes them as its lowest level): {len(nd)}, "
      f"{nd.employment_est.sum():,.0f} jobs in 2025, degree-reachable among them: {int(nd.is_degree_entry.sum())}")

tot = (est.assign(group=np.where(est.is_degree_entry, "degree-reachable", "other entry"))
          .pivot_table(index="group", columns="year", values="employment_est", aggfunc="sum", margins=True, margins_name="all jobs"))
tot_share = tot.div(tot.loc["all jobs"], axis=1)
show(tot.round(0).astype(int), "Estimated NYC jobs by year (sum of the D8 estimate; equals QCEW NYC total)")
show((tot_share * 100).round(1), "Share of NYC jobs, %")
chg = pd.DataFrame({"2019": tot[2019], "2025": tot[2025], "change": tot[2025] - tot[2019],
                    "growth %": (tot[2025] / tot[2019] - 1) * 100}).round(1)
show(chg, "2019 → 2025")

fct_job_employment_nyc_est rows read: 5,357 (815 jobs × 7 years)
estimated codes that are broad/aggregate rows in dim_job (OEWS research publishes them as its lowest level): 12, 480,169 jobs in 2025, degree-reachable among them: 4


**Estimated NYC jobs by year (sum of the D8 estimate; equals QCEW NYC total)**

year,2019,2020,2021,2022,2023,2024,2025,all jobs
group,,,,,,,,
degree-reachable,1216185,1168227,1242281,1330230,1361117,1413723,1445722,9177485
other entry,3276526,2829531,2823498,3017131,3124578,3216473,3207207,21494944
all jobs,4492711,3997758,4065779,4347361,4485695,4630196,4652929,30672429


**Share of NYC jobs, %**

year,2019,2020,2021,2022,2023,2024,2025,all jobs
group,,,,,,,,
degree-reachable,27.10,29.20,30.60,30.60,30.30,30.50,31.10,29.90
other entry,72.90,70.80,69.40,69.40,69.70,69.50,68.90,70.10
all jobs,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00


**2019 → 2025**

,2019,2025,change,growth %
group,,,,
degree-reachable,"1,216,185.30","1,445,722.20","229,536.90",18.90
other entry,"3,276,525.70","3,207,206.80","-69,318.90",-2.10
all jobs,"4,492,711.00","4,652,929.00","160,218.00",3.60


The same split on the published metro series, to see whether the estimate's direction is the metro's direction.

In [4]:
metro = q("""
select year, soc_code, employment, quality_flag, series_note
from fct_job_employment_wage_annual where occ_level = 'detailed' and year between 2019 and 2025
""").merge(jobs[["soc_code", "is_degree_entry"]], on="soc_code", how="inner")
mt = (metro.assign(group=np.where(metro.is_degree_entry, "degree-reachable", "other entry"))
           .pivot_table(index="group", columns="year", values="employment", aggfunc="sum", margins=True, margins_name="all detailed"))
show(pd.DataFrame({"2019": mt[2019], "2025": mt[2025], "growth %": (mt[2025] / mt[2019] - 1) * 100}).round(1),
     "NY metro (OEWS, published), detailed rows only")

**NY metro (OEWS, published), detailed rows only**

,2019,2025,growth %
group,,,
degree-reachable,"2,436,919.40","2,678,790.00",9.90
other entry,"6,445,200.60","5,816,500.00",-9.80
all detailed,"8,882,120.00","8,495,290.00",-4.40


### By SOC major group (degree-reachable jobs only)

In [5]:
grp = (est[est.is_degree_entry].pivot_table(index=["major_code", "major_title"], columns="year", values="employment_est", aggfunc="sum"))
grp_out = pd.DataFrame({"jobs": est[(est.is_degree_entry) & (est.year == 2025)].groupby(["major_code", "major_title"]).size(),
                        "2019": grp[2019], "2025": grp[2025], "change": grp[2025] - grp[2019],
                        "growth %": (grp[2025] / grp[2019] - 1) * 100}).sort_values("2025", ascending=False)
show(grp_out.round(1), "Degree-reachable NYC jobs by SOC major group, 2019 → 2025 (estimate)")

**Degree-reachable NYC jobs by SOC major group, 2019 → 2025 (estimate)**

,,jobs,2019,2025,change,growth %
major_code,major_title,,,,,
13-0000,Business and Financial Operations Occupations,28,"328,724.90","390,345.20","61,620.30",18.70
11-0000,Management Occupations,27,"213,211.20","330,479.80","117,268.60",55.00
15-0000,Computer and Mathematical Occupations,17,"127,851.80","160,302.40","32,450.60",25.40
29-0000,Healthcare Practitioners and Technical Occupations,18,"127,807.30","152,286.30","24,479.00",19.20
25-0000,Educational Instruction and Library Occupations,19,"123,752.00","135,087.20","11,335.20",9.20
27-0000,"Arts, Design, Entertainment, Sports, and Media Occupations",22,"103,586.80","93,267.30","-10,319.50",-10.00
17-0000,Architecture and Engineering Occupations,32,"46,085.30","44,286.50","-1,798.80",-3.90
41-0000,Sales and Related Occupations,3,"52,365.70","43,234.70","-9,131.00",-17.40
21-0000,Community and Social Service Occupations,8,"35,659.90","36,291.90",632.00,1.80


## 3. List 1: most openings (projected 2022-32)

Leaf rows of the 2022-32 NYC edition (`is_leaf`), degree-reachable, All Other codes set aside. Openings = growth + separations averaged over ten years. `quality_flag = 'imputed'` marks the D1/D15 fill rows. Entry wage = NYS DOL NYC wage table (`fct_job_wage_nyc`, survey May 2025 aged to Q1 2026); `prospects` = NYS DOL's own Employment Prospects rating from the same projections release (an openings-quartile × growth-band matrix).

In [6]:
proj = q("""
select p.soc_code, p.base_employment, p.projected_employment, p.pct_change, p.annual_openings_avg, p.quality_flag
from fct_job_projection p where p.edition = '2022-2032' and p.is_leaf
""")
wage = q("select soc_code, entry_wage, median_wage, entry_wage_at_floor from fct_job_wage_nyc")
prospects = q("""
select soc_code, employment_prospects as prospects
from staging.stg_nysdol__projections_xlsx where edition = '2022-2032' and prospects_usable
""")
alljob = q("select soc_code, soc_title, soc_level, is_degree_entry, is_all_other, entry_education from dim_job")
print(f"projection leaves: {len(proj):,} | wage rows: {len(wage):,} | prospects rows: {len(prospects):,}")

P = (proj.merge(alljob, on="soc_code", how="left").merge(wage, on="soc_code", how="left").merge(prospects, on="soc_code", how="left"))
assert P.soc_title.notna().all()
Pd = P[P.is_degree_entry].copy()
print(f"degree-reachable leaves: {len(Pd)} (All Other: {int(Pd.is_all_other.sum())}); "
      f"openings {Pd.annual_openings_avg.sum():,.0f} of {P.annual_openings_avg.sum():,.0f} = {Pd.annual_openings_avg.sum() / P.annual_openings_avg.sum():.1%} of all NYC openings")

cols1 = ["soc_code", "soc_title", "entry_education", "base_employment", "pct_change", "annual_openings_avg", "entry_wage", "median_wage", "prospects", "quality_flag"]
top_open = Pd[~Pd.is_all_other].nlargest(25, "annual_openings_avg")[cols1].reset_index(drop=True)
top_open.index += 1
show(top_open, "List 1 · Most projected annual openings, 2022-32, degree-reachable (All Other excluded)")
show(Pd[Pd.is_all_other].nlargest(8, "annual_openings_avg")[cols1].reset_index(drop=True), "“Other jobs” list: All Other codes with the most openings")

projection leaves: 592 | wage rows: 702 | prospects rows: 646
degree-reachable leaves: 186 (All Other: 22); openings 136,787 of 599,218 = 22.8% of all NYC openings


**List 1 · Most projected annual openings, 2022-32, degree-reachable (All Other excluded)**

,soc_code,soc_title,entry_education,base_employment,pct_change,annual_openings_avg,entry_wage,median_wage,prospects,quality_flag
1,11-1021,General and Operations Managers,Bachelor's degree,"107,930.00",8.63,"9,781.00","83,885.00","160,470.00",Favorable,published
2,29-1141,Registered Nurses,Bachelor's degree,"86,760.00",24.90,"7,410.00","95,811.00","127,613.00",Very Favorable,published
3,13-2011,Accountants and Auditors,Bachelor's degree,"76,330.00",7.97,"6,648.00","78,988.00","110,746.00",Favorable,published
4,13-1161,Market Research Analysts and Marketing Specialists,Bachelor's degree,"58,570.00",9.34,"6,037.00","68,400.00","103,602.00",Favorable,published
5,13-1111,Management Analysts,Bachelor's degree,"45,700.00",11.90,"4,444.00","84,016.00","130,719.00",Favorable,published
6,15-1252,Software Developers,Bachelor's degree,"64,226.20",11.25,"4,308.19","121,283.00","174,663.00",NaN,imputed
7,41-3031,"Securities, Commodities, and Financial Services Sales ...",Bachelor's degree,"57,520.00",-0.78,"4,145.00","97,955.00","175,416.00",Unfavorable,published
8,25-2021,"Elementary School Teachers, Except Special Education",Bachelor's degree,"39,390.00",18.35,"3,573.00","66,576.00","103,343.00",Very Favorable,published
9,11-3031,Financial Managers,Bachelor's degree,"37,390.00",16.96,"3,334.00","161,264.00","242,789.00",Very Favorable,published
10,13-1071,Human Resources Specialists,Bachelor's degree,"30,300.00",14.49,"3,099.00","63,824.00","97,527.00",Very Favorable,published


**“Other jobs” list: All Other codes with the most openings**

,soc_code,soc_title,entry_education,base_employment,pct_change,annual_openings_avg,entry_wage,median_wage,prospects,quality_flag
0,13-1199,"Business Operations Specialists, All Other",Bachelor's degree,"20,245.86",8.50,"2,023.85","55,732.00","91,532.00",NaN,imputed
1,11-9199,"Managers, All Other",Bachelor's degree,"24,763.50",1.41,858.84,"115,735.00","181,934.00",NaN,imputed
2,21-1099,"Community and Social Service Specialists, All Other",Bachelor's degree,"6,360.00",20.44,820.00,"58,618.00","68,448.00",Very Favorable,published
3,13-2099,"Financial Specialists, All Other",Bachelor's degree,"8,068.39",2.26,563.10,"76,033.00","132,834.00",NaN,imputed
4,19-4099,"Life, Physical, and Social Science Technicians, All Other",Associate's degree,"2,660.00",16.54,394.00,"53,425.00","65,591.00",Favorable,published
5,25-3099,"Teachers and Instructors, All Other",Bachelor's degree,"2,403.46",15.77,299.00,"38,964.00","83,395.00",NaN,imputed
6,15-1299,"Computer Occupations, All Other",Bachelor's degree,"3,870.00",7.49,279.00,"77,227.00","134,670.00",Favorable,published
7,21-2099,"Religious Workers, All Other",Bachelor's degree,"1,540.00",18.83,249.00,"35,546.00","52,997.00",Favorable,published


## 4. List 2: fastest actual growth, 2019 → 2025 (NYC estimate), floor 1,000 jobs

Growth is read from the D8 estimate, with the metro published growth and the projected 2022-32 growth beside it. The floor (plan §3b) removes jobs where a suppressed pattern cell, not the labor market, moves the series. `split_2019` marks jobs whose 2019 value was carried over from a SOC 2010 / hybrid code (reclassification is part of their growth).

In [7]:
E = est.pivot_table(index="soc_code", columns="year", values="employment_est")
M = est.pivot_table(index="soc_code", columns="year", values="metro_employment")
split19 = set(metro.loc[(metro.year == 2019) & (metro.quality_flag != "published"), "soc_code"])
G = pd.DataFrame({"nyc_2019": E[2019], "nyc_2025": E[2025], "metro_2019": M[2019], "metro_2025": M[2025]})
G["nyc_growth %"] = (G.nyc_2025 / G.nyc_2019 - 1) * 100
G["metro_growth %"] = (G.metro_2025 / G.metro_2019 - 1) * 100
G["split_2019"] = G.index.isin(split19)
G = (G.reset_index().merge(jobs_all[["soc_code", "soc_title", "entry_education", "is_degree_entry", "is_all_other"]], on="soc_code")
      .merge(proj[["soc_code", "pct_change", "annual_openings_avg", "quality_flag"]].rename(columns={"pct_change": "proj_2022_32 %"}), on="soc_code", how="left")
      .merge(wage, on="soc_code", how="left").merge(prospects, on="soc_code", how="left"))
Gd = G[G.is_degree_entry & G.nyc_2019.notna() & G.nyc_2025.notna()].copy()
floor = Gd[(Gd.nyc_2019 >= 1000) & (Gd.nyc_2025 >= 1000)]
print(f"degree-reachable jobs with 2019 and 2025 estimates: {len(Gd)} | clearing the 1,000 floor in both years: {len(floor)} "
      f"({floor.nyc_2025.sum() / Gd.nyc_2025.sum():.1%} of their 2025 jobs) | All Other among them: {int(floor.is_all_other.sum())}")

cols2 = ["soc_code", "soc_title", "entry_education", "nyc_2019", "nyc_2025", "nyc_growth %", "metro_growth %", "proj_2022_32 %",
         "annual_openings_avg", "entry_wage", "prospects", "split_2019"]
top_grow = floor[~floor.is_all_other].nlargest(25, "nyc_growth %")[cols2].reset_index(drop=True)
top_grow.index += 1
show(top_grow, "List 2 · Fastest growing 2019 → 2025, degree-reachable, ≥ 1,000 NYC jobs (All Other excluded)")
show(floor[~floor.is_all_other].nsmallest(15, "nyc_growth %")[cols2].reset_index(drop=True), "Fastest shrinking, same universe")

degree-reachable jobs with 2019 and 2025 estimates: 212 | clearing the 1,000 floor in both years: 132 (97.2% of their 2025 jobs) | All Other among them: 11


**List 2 · Fastest growing 2019 → 2025, degree-reachable, ≥ 1,000 NYC jobs (All Other excluded)**

,soc_code,soc_title,entry_education,nyc_2019,nyc_2025,nyc_growth %,metro_growth %,proj_2022_32 %,annual_openings_avg,entry_wage,prospects,split_2019
1,15-2051,Data Scientists,Bachelor's degree,"1,094.48","14,412.05","1,216.79",763.57,33.33,811.38,"89,947.00",NaN,True
2,11-2021,Marketing Managers,Bachelor's degree,"12,204.46","31,461.21",157.78,93.80,7.50,"1,857.00","133,848.00",Favorable,False
3,13-1082,Project Management Specialists,Bachelor's degree,"16,703.32","34,711.01",107.81,51.80,11.43,"2,917.77","85,707.00",NaN,True
4,13-1081,Logisticians,Bachelor's degree,"1,494.52","3,004.74",101.05,-2.74,22.06,225.00,"66,346.00",Very Favorable,False
5,11-3031,Financial Managers,Bachelor's degree,"27,361.24","49,761.01",81.87,59.82,16.96,"3,334.00","161,264.00",Very Favorable,False
6,11-3021,Computer and Information Systems Managers,Bachelor's degree,"16,576.02","28,787.62",73.67,75.61,14.19,"1,848.00","164,322.00",Favorable,False
7,15-1252,Software Developers,Bachelor's degree,"41,641.48","70,056.41",68.24,31.84,11.25,"4,308.19","121,283.00",NaN,True
8,11-3121,Human Resources Managers,Bachelor's degree,"5,909.43","9,897.07",67.48,53.74,12.20,681.00,"127,224.00",Favorable,False
9,25-2032,"Career/Technical Education Teachers, Secondary School",Bachelor's degree,"1,586.98","2,651.66",67.09,53.82,18.58,101.00,"68,918.00",Favorable,False
10,15-1253,Software Quality Assurance Analysts and Testers,Bachelor's degree,"3,943.90","6,544.25",65.93,31.84,6.71,419.49,"91,352.00",NaN,True


**Fastest shrinking, same universe**

,soc_code,soc_title,entry_education,nyc_2019,nyc_2025,nyc_growth %,metro_growth %,proj_2022_32 %,annual_openings_avg,entry_wage,prospects,split_2019
0,27-4012,Broadcast Technicians,Associate's degree,"3,409.76","1,231.38",-63.89,-50.49,-3.05,338.00,"49,335.00",Very Unfavorable,False
1,15-1251,Computer Programmers,Bachelor's degree,"7,219.50","3,140.37",-56.50,-55.74,-8.40,325.00,"91,963.00",Very Unfavorable,False
2,11-2011,Advertising and Promotions Managers,Bachelor's degree,"4,767.09","2,141.63",-55.07,-53.64,9.05,725.00,"104,573.00",Favorable,False
3,19-4061,Social Science Research Assistants,Bachelor's degree,"3,595.03","1,736.72",-51.69,-32.56,9.86,291.00,"45,999.00",Favorable,False
4,15-1231,Computer Network Support Specialists,Associate's degree,"7,812.85","3,982.13",-49.03,-52.94,8.11,459.00,"59,341.00",Favorable,False
5,15-1211,Computer Systems Analysts,Bachelor's degree,"23,213.50","13,541.70",-41.66,-39.97,12.13,"1,090.00","88,440.00",Favorable,False
6,15-1254,Web Developers,Bachelor's degree,"1,680.80","1,010.52",-39.88,42.57,4.79,163.46,"57,962.00",NaN,True
7,27-1014,Special Effects Artists and Animators,Bachelor's degree,"1,714.60","1,043.73",-39.13,-42.22,1.28,223.00,"71,377.00",Unfavorable,False
8,27-1024,Graphic Designers,Bachelor's degree,"11,558.80","7,372.24",-36.22,-30.59,3.95,"1,279.00","58,304.00",Favorable,False
9,17-3023,Electrical and Electronic Engineering Technologists an...,Associate's degree,"2,078.20","1,331.18",-35.95,-30.52,3.66,83.00,"64,117.00",Unfavorable,False


## 5. Checks

### 5a. Do the NYC estimate and the metro series agree on growth?
Two independent sources: QCEW × staffing pattern vs the OEWS metro survey. Rank agreement across the floor universe, and the jobs where they disagree on the sign.

In [8]:
f = floor.dropna(subset=["metro_growth %"])
print(f"jobs compared: {len(f)} | Pearson r = {f['nyc_growth %'].corr(f['metro_growth %']):.2f} | "
      f"Spearman ρ = {f['nyc_growth %'].rank().corr(f['metro_growth %'].rank()):.2f}")
sign_flip = f[np.sign(f["nyc_growth %"]) != np.sign(f["metro_growth %"])]
print(f"sign disagreements: {len(sign_flip)} of {len(f)}")
show(sign_flip.assign(gap=lambda d: (d["nyc_growth %"] - d["metro_growth %"]).abs()).nlargest(12, "gap")[cols2[:8]].reset_index(drop=True),
     "Largest sign disagreements (NYC estimate vs metro published)")
big_gap = f[(f["nyc_growth %"] - f["metro_growth %"]).abs() > 30]
show(big_gap[cols2[:8]].sort_values("nyc_growth %", ascending=False).reset_index(drop=True), "Gaps above 30 points, either sign")

jobs compared: 132 | Pearson r = 0.97 | Spearman ρ = 0.83
sign disagreements: 23 of 132


**Largest sign disagreements (NYC estimate vs metro published)**

,soc_code,soc_title,entry_education,nyc_2019,nyc_2025,nyc_growth %,metro_growth %,proj_2022_32 %
0,13-1081,Logisticians,Bachelor's degree,"1,494.52","3,004.74",101.05,-2.74,22.06
1,15-1254,Web Developers,Bachelor's degree,"1,680.80","1,010.52",-39.88,42.57,4.79
2,13-2020,Property Appraisers and Assessors,Bachelor's degree,"1,443.74","1,131.19",-21.65,17.73,6.63
3,11-1011,Chief Executives,Bachelor's degree,"4,613.73","3,577.37",-22.46,11.66,NaN
4,25-3031,"Substitute Teachers, Short-Term",Bachelor's degree,"11,296.61","13,348.68",18.17,-11.80,22.18
5,25-3099,"Teachers and Instructors, All Other",Bachelor's degree,"1,290.00","1,436.36",11.35,-18.09,15.77
6,17-2081,Environmental Engineers,Bachelor's degree,"1,715.48","1,815.44",5.83,-20.26,18.37
7,25-2051,"Special Education Teachers, Preschool",Bachelor's degree,"2,811.24","2,583.55",-8.10,17.05,15.57
8,19-2031,Chemists,Bachelor's degree,"1,313.42","1,382.77",5.28,-18.18,13.95
9,17-2112,Industrial Engineers,Bachelor's degree,"2,910.66","2,570.89",-11.67,10.48,10.32


**Gaps above 30 points, either sign**

,soc_code,soc_title,entry_education,nyc_2019,nyc_2025,nyc_growth %,metro_growth %,proj_2022_32 %
0,15-2051,Data Scientists,Bachelor's degree,"1,094.48","14,412.05","1,216.79",763.57,33.33
1,15-1299,"Computer Occupations, All Other",Bachelor's degree,"2,337.55","6,661.01",184.96,33.58,7.49
2,11-2021,Marketing Managers,Bachelor's degree,"12,204.46","31,461.21",157.78,93.80,7.50
3,13-1082,Project Management Specialists,Bachelor's degree,"16,703.32","34,711.01",107.81,51.80,11.43
4,13-1081,Logisticians,Bachelor's degree,"1,494.52","3,004.74",101.05,-2.74,22.06
5,15-1252,Software Developers,Bachelor's degree,"41,641.48","70,056.41",68.24,31.84,11.25
6,15-1253,Software Quality Assurance Analysts and Testers,Bachelor's degree,"3,943.90","6,544.25",65.93,31.84,6.71
7,11-9199,"Managers, All Other",Bachelor's degree,"8,554.85","10,259.71",19.93,59.43,1.41
8,29-2031,Cardiovascular Technologists and Technicians,Associate's degree,"1,665.58","1,855.72",11.42,46.15,24.16
9,13-2020,Property Appraisers and Assessors,Bachelor's degree,"1,443.74","1,131.19",-21.65,17.73,6.63


### 5b. Do the lists agree with NYS DOL's Employment Prospects?
NYS DOL rates every job from the same projections (openings quartile × growth band). Our List 1 should sit in Favorable / Very Favorable almost by construction (openings are an input); List 2 is actual growth, so disagreement there is information, not error.

In [9]:
def xtab(df, label):
    t = df.prospects.fillna("(not rated)").value_counts().rename(label)
    return t
show(pd.concat([xtab(top_open, "List 1 (openings)"), xtab(top_grow, "List 2 (growth)"),
                xtab(Pd[~Pd.is_all_other], "all degree-reachable leaves")], axis=1).fillna(0).astype(int)
       .reindex(["Very Favorable", "Favorable", "Unfavorable", "Very Unfavorable", "(not rated)"]),
     "NYS DOL Employment Prospects for the jobs on each list")
show(top_grow[top_grow.prospects.isin(["Unfavorable", "Very Unfavorable"])][cols2[:8] + ["prospects"]].reset_index(drop=True),
     "List 2 jobs that NYS DOL rates Unfavorable (actual growth vs projected outlook)")

**NYS DOL Employment Prospects for the jobs on each list**

,List 1 (openings),List 2 (growth),all degree-reachable leaves
prospects,,,
Very Favorable,9,9,50
Favorable,11,8,67
Unfavorable,2,1,20
Very Unfavorable,0,0,9
(not rated),3,7,18


**List 2 jobs that NYS DOL rates Unfavorable (actual growth vs projected outlook)**

,soc_code,soc_title,entry_education,nyc_2019,nyc_2025,nyc_growth %,metro_growth %,proj_2022_32 %,prospects
0,41-9031,Sales Engineers,Bachelor's degree,"1,278.64","1,626.76",27.23,39.27,4.82,Unfavorable


### 5c. Why the floor: the swings below 1,000 jobs
Year-over-year moves of more than 50% in the estimate while the metro series moved less than 15%. If the floor is right, every case is small.

In [10]:
yy = est.sort_values(["soc_code", "year"]).copy()
yy["est_prev"] = yy.groupby("soc_code").employment_est.shift()
yy["metro_prev"] = yy.groupby("soc_code").metro_employment.shift()
yy["ns_prev"] = yy.groupby("soc_code").n_sectors_contributing.shift()
yy["g_est"] = yy.employment_est / yy.est_prev - 1
yy["g_metro"] = yy.metro_employment / yy.metro_prev - 1
sw = yy[yy.is_degree_entry & (yy.g_est.abs() > 0.5) & (yy.g_metro.abs() < 0.15)]
print(f"swing cases among degree-reachable jobs, 2020–2025: {len(sw)} | largest previous-year estimate among them: {sw.est_prev.max():,.0f}")
show(sw[["year", "soc_code", "major_title", "est_prev", "employment_est", "g_est", "g_metro", "ns_prev", "n_sectors_contributing"]]
       .merge(jobs[["soc_code", "soc_title"]], on="soc_code").sort_values(["year", "soc_code"]).reset_index(drop=True),
     "Swing cases (all under the floor)")

swing cases among degree-reachable jobs, 2020–2025: 22 | largest previous-year estimate among them: 752


**Swing cases (all under the floor)**

,year,soc_code,major_title,est_prev,employment_est,g_est,g_metro,ns_prev,n_sectors_contributing,soc_title
0,2020,17-3012,Architecture and Engineering Occupations,203.53,334.46,0.64,-0.05,3.00,4,Electrical and Electronics Drafters
1,2020,17-3024,Architecture and Engineering Occupations,33.76,80.59,1.39,0.14,1.00,2,Electro-Mechanical and Mechatronics Technologists and ...
2,2020,17-3026,Architecture and Engineering Occupations,417.58,649.20,0.55,-0.11,4.00,3,Industrial Engineering Technologists and Technicians
3,2020,19-1012,"Life, Physical, and Social Science Occupations",168.83,42.24,-0.75,-0.07,3.00,2,Food Scientists and Technologists
4,2020,19-2021,"Life, Physical, and Social Science Occupations",126.98,221.20,0.74,0.08,2.00,3,Atmospheric and Space Scientists
5,2021,17-2041,Architecture and Engineering Occupations,288.66,97.83,-0.66,-0.04,3.00,2,Chemical Engineers
6,2021,17-2121,Architecture and Engineering Occupations,55.86,17.68,-0.68,-0.11,1.00,1,Marine Engineers and Naval Architects
7,2021,17-3024,Architecture and Engineering Occupations,80.59,23.22,-0.71,0.08,2.00,1,Electro-Mechanical and Mechatronics Technologists and ...
8,2022,17-2041,Architecture and Engineering Occupations,97.83,352.97,2.61,-0.14,2.00,3,Chemical Engineers
9,2022,17-2061,Architecture and Engineering Occupations,585.93,"1,045.02",0.78,0.12,5.00,5,Computer Hardware Engineers


### 5d. Trace one job by hand: Registered Nurses `29-1141`
Recomputed from the pattern audit table: Σ over sectors of NYC sector employment × the job's share of that sector, 2025.

In [11]:
rn = q("""
select a.pattern_sector, a.share_of_sector, s.employment as sector_employment, a.share_of_sector * s.employment as contribution
from audit_industry_job_pattern_annual a
join audit_nyc_employment_by_pattern_sector s on s.year = a.year and s.pattern_sector = a.pattern_sector
where a.year = 2025 and a.soc_code = '29-1141' order by contribution desc
""")
show(rn.round(4), "RN 2025: sector contributions")
fact_rn = est[(est.soc_code == "29-1141") & (est.year == 2025)].iloc[0]
print(f"Σ contributions = {rn.contribution.sum():,.0f} | fact employment_est = {fact_rn.employment_est:,.0f} | "
      f"NYS DOL NYC count (May 2025) = {fact_rn.nyc_wage_table_employment:,.0f} | metro = {fact_rn.metro_employment:,.0f}")
assert abs(rn.contribution.sum() - fact_rn.employment_est) < 1

**RN 2025: sector contributions**

,pattern_sector,share_of_sector,sector_employment,contribution
0,62,0.08,"1,054,936.29","87,038.38"
1,56,0.03,"238,646.25","7,900.67"
2,99,0.01,"360,757.00","4,688.32"
3,61,0.01,"354,371.09","3,989.68"
4,52,0.01,"363,372.58","3,415.52"
5,55,0.01,"78,980.13",652.56
6,54,0.00,"433,780.20",611.04
7,81,0.00,"163,866.37",216.41
8,42,0.00,"123,426.82",16.73
9,31-33,0.00,"52,116.99",5.16


Σ contributions = 108,534 | fact employment_est = 108,534 | NYS DOL NYC count (May 2025) = 89,560 | metro = 197,740


### 5e. Row accounting
Every table above against the fact it came from.

In [12]:
acct = pd.DataFrame([
    ["dim_job detailed", len(jobs), "221 degree-reachable", int(jobs.is_degree_entry.sum())],
    ["fct_job_employment_nyc_est", len(est), "jobs with 2019 and 2025, degree-reachable", len(Gd)],
    ["  … clearing the floor", len(floor), "of which All Other", int(floor.is_all_other.sum())],
    ["fct_job_projection 2022-32 leaves", len(proj), "degree-reachable leaves", len(Pd)],
    ["fct_job_wage_nyc", len(wage), "List 1 rows with an entry wage", int(top_open.entry_wage.notna().sum())],
    ["prospects (usable)", len(prospects), "List 2 rows rated", int(top_grow.prospects.notna().sum())],
], columns=["read", "rows", "kept as", "rows kept"])
show(acct, "Row accounting")

**Row accounting**

,read,rows,kept as,rows kept
0,dim_job detailed,867,221 degree-reachable,221
1,fct_job_employment_nyc_est,5357,"jobs with 2019 and 2025, degree-reachable",212
2,… clearing the floor,132,of which All Other,11
3,fct_job_projection 2022-32 leaves,592,degree-reachable leaves,186
4,fct_job_wage_nyc,702,List 1 rows with an entry wage,25
5,prospects (usable),646,List 2 rows rated,18


## 6. Featured areas (S7): the four areas 2019 → 2025
Degree-reachable jobs flagged `in_tech` / `in_health` / `in_business` / `in_education` in `dim_job` (Career Clusters, D11). A job can sit in more than one area, so the rows do not add.

In [13]:
rows = []
for area, col in [("Tech", "in_tech"), ("Healthcare", "in_health"), ("Business", "in_business"), ("Education", "in_education")]:
    area_codes = jobs_all.loc[jobs_all[col].fillna(False).astype(bool), "soc_code"]
    sub = Gd[Gd.soc_code.isin(area_codes)]
    pr = Pd[Pd.soc_code.isin(area_codes)]
    wgrowth = (pr["pct_change"] * pr.base_employment).sum() / pr.base_employment.sum() if len(pr) else np.nan
    rows.append([area, len(sub), sub.nyc_2019.sum(), sub.nyc_2025.sum(), (sub.nyc_2025.sum() / sub.nyc_2019.sum() - 1) * 100,
                 pr.annual_openings_avg.sum(), wgrowth])
show(pd.DataFrame(rows, columns=["area", "jobs", "nyc_2019", "nyc_2025", "growth %", "openings 2022-32", "projected growth % (emp-weighted)"]).round(1),
     "Featured areas, degree-reachable jobs")

**Featured areas, degree-reachable jobs**

,area,jobs,nyc_2019,nyc_2025,growth %,openings 2022-32,projected growth % (emp-weighted)
0,Tech,16,"139,368.20","184,791.10",32.60,"12,468.40",11.10
1,Healthcare,40,"197,549.10","232,705.00",17.80,"18,270.20",25.20
2,Business,48,"558,404.40","697,820.60",25.00,"68,629.60",9.10
3,Education,25,"185,423.40","202,924.20",9.40,"23,463.00",17.20


## 7. Why the openings exist: growth vs people leaving, and the churn of the job's industries

**Rule:** plan §3g, written before this ran. NYS DOL's `annual_openings_avg` = `growth_openings_avg` + `separation_openings_avg`; separations are labor force exits plus occupational transfers (the BLS separations method), never employer changes inside the job, and negative growth is netted, so a shrinking job's openings sit below its separations. Growth share = growth ÷ total; bands: shrinking (< 0), mostly replacement (0 to the all-leaves share 9.5%), mixed (to 25%), growth-led (≥ 25%).

Beside it, how churny the job's industries are: QWI private new-hire and separation rates by sector (five boroughs summed, all ages, 2024 = last complete year; 2019 beside), weighted by the job's sector mix (`bridge_industry_job.share_of_job`). QWI has no occupation, so this describes the industries the job sits in, not the job's own turnover.

In [14]:
split = q("""
select soc_code, annual_openings_avg, growth_openings_avg, separation_openings_avg
from fct_job_projection where edition = '2022-2032' and is_leaf
""")
ident_off = int(((split.annual_openings_avg - split.growth_openings_avg - split.separation_openings_avg).abs() > 1).sum())
all_share = split.growth_openings_avg.sum() / split.annual_openings_avg.sum()
split["growth_share"] = split.growth_openings_avg / split.annual_openings_avg
print(f"leaves: {len(split)} | rows where openings ≠ growth + separations by more than 1: {ident_off} | leaves with negative growth: {int((split.growth_openings_avg < 0).sum())} "
      f"(min {split.growth_openings_avg.min():,.0f}) | all-leaves growth share: {all_share:.1%} (plan §3g cutoff)")
assert ident_off == 0

S = Pd.merge(split[["soc_code", "growth_openings_avg", "separation_openings_avg", "growth_share"]], on="soc_code", how="left")
assert S.growth_share.notna().all()
S["openings_from"] = np.select([S.growth_share < 0, S.growth_share < all_share, S.growth_share < 0.25],
                               ["shrinking", "mostly replacement", "mixed"], default="growth-led")
band_order = ["growth-led", "mixed", "mostly replacement", "shrinking"]
deg_share = S.growth_openings_avg.sum() / S.annual_openings_avg.sum()
print(f"degree-reachable leaves: growth openings {S.growth_openings_avg.sum():,.0f} of {S.annual_openings_avg.sum():,.0f} = {deg_share:.1%} "
      f"(vs {all_share:.1%} for every leaf); separations {S.separation_openings_avg.sum():,.0f}")
bands = (S.groupby("openings_from").agg(jobs=("soc_code", "size"), openings=("annual_openings_avg", "sum"), growth=("growth_openings_avg", "sum"),
                                       separations=("separation_openings_avg", "sum"), base_employment=("base_employment", "sum"))
          .reindex(band_order))
bands["share of openings %"] = bands.openings / bands.openings.sum() * 100
show(bands.round(1), "Degree-reachable leaves by where their openings come from (plan §3g bands)")

leaves: 592 | rows where openings ≠ growth + separations by more than 1: 0 | leaves with negative growth: 86 (min -727) | all-leaves growth share: 9.7% (plan §3g cutoff)
degree-reachable leaves: growth openings 17,894 of 136,787 = 13.1% (vs 9.7% for every leaf); separations 118,893


**Degree-reachable leaves by where their openings come from (plan §3g bands)**

,jobs,openings,growth,separations,base_employment,share of openings %
openings_from,,,,,,
growth-led,15,"13,161.30","4,009.40","9,151.90","142,746.40",9.60
mixed,110,"64,922.00","10,667.80","54,254.20","683,445.10",47.50
mostly replacement,47,"49,128.50","3,592.00","45,536.50","574,029.30",35.90
shrinking,14,"9,575.00",-375.00,"9,950.00","122,310.00",7.00


In [15]:
S["growth share %"] = S.growth_share * 100
cols7 = ["soc_code", "soc_title", "annual_openings_avg", "growth_openings_avg", "separation_openings_avg", "growth share %", "openings_from", "pct_change", "entry_wage"]
list1 = S[S.soc_code.isin(top_open.soc_code)].set_index("soc_code").reindex(top_open.soc_code).reset_index()
list1.index += 1
show(list1[cols7].round(1), "List 1 again, with the growth / replacement split")
show(S[~S.is_all_other & (S.openings_from == "growth-led")].nlargest(15, "annual_openings_avg")[cols7].round(1).reset_index(drop=True),
     "Growth-led degree-reachable jobs with the most openings")
show(S[~S.is_all_other & (S.openings_from == "shrinking")].nlargest(12, "annual_openings_avg")[cols7].round(1).reset_index(drop=True),
     "Shrinking jobs that still open (replacement only, fewer than their separations)")

**List 1 again, with the growth / replacement split**

,soc_code,soc_title,annual_openings_avg,growth_openings_avg,separation_openings_avg,growth share %,openings_from,pct_change,entry_wage
1,11-1021,General and Operations Managers,"9,781.00",931.00,"8,850.00",9.50,mostly replacement,8.60,"83,885.00"
2,29-1141,Registered Nurses,"7,410.00","2,160.00","5,250.00",29.10,growth-led,24.90,"95,811.00"
3,13-2011,Accountants and Auditors,"6,648.00",608.00,"6,040.00",9.10,mostly replacement,8.00,"78,988.00"
4,13-1161,Market Research Analysts and Marketing Specialists,"6,037.00",547.00,"5,490.00",9.10,mostly replacement,9.30,"68,400.00"
5,13-1111,Management Analysts,"4,444.00",544.00,"3,900.00",12.20,mixed,11.90,"84,016.00"
6,15-1252,Software Developers,"4,308.20",722.80,"3,585.40",16.80,mixed,11.30,"121,283.00"
7,41-3031,"Securities, Commodities, and Financial Services Sales ...","4,145.00",-45.00,"4,190.00",-1.10,shrinking,-0.80,"97,955.00"
8,25-2021,"Elementary School Teachers, Except Special Education","3,573.00",723.00,"2,850.00",20.20,mixed,18.40,"66,576.00"
9,11-3031,Financial Managers,"3,334.00",634.00,"2,700.00",19.00,mixed,17.00,"161,264.00"
10,13-1071,Human Resources Specialists,"3,099.00",439.00,"2,660.00",14.20,mixed,14.50,"63,824.00"


**Growth-led degree-reachable jobs with the most openings**

,soc_code,soc_title,annual_openings_avg,growth_openings_avg,separation_openings_avg,growth share %,openings_from,pct_change,entry_wage
0,29-1141,Registered Nurses,"7,410.00","2,160.00","5,250.00",29.10,growth-led,24.90,"95,811.00"
1,11-9111,Medical and Health Services Managers,"1,738.00",618.00,"1,120.00",35.60,growth-led,47.20,"115,316.00"
2,15-2051,Data Scientists,811.40,267.50,543.90,33.00,growth-led,33.30,"89,947.00"
3,29-2010,Clinical Laboratory Technologists and Technicians,808.00,208.00,600.00,25.70,growth-led,24.70,"62,123.00"
4,29-2034,Radiologic Technologists and Technicians,607.00,187.00,420.00,30.80,growth-led,26.20,"85,719.00"
5,15-1212,Information Security Analysts,547.00,167.00,380.00,30.50,growth-led,29.20,"101,787.00"
6,29-1292,Dental Hygienists,320.00,80.00,240.00,25.00,growth-led,23.30,"91,402.00"
7,29-2032,Diagnostic Medical Sonographers,273.00,103.00,170.00,37.70,growth-led,36.50,"93,202.00"
8,29-1126,Respiratory Therapists,271.00,101.00,170.00,37.30,growth-led,34.70,"100,800.00"
9,15-2011,Actuaries,130.00,40.00,90.00,30.80,growth-led,23.50,"103,699.00"


**Shrinking jobs that still open (replacement only, fewer than their separations)**

,soc_code,soc_title,annual_openings_avg,growth_openings_avg,separation_openings_avg,growth share %,openings_from,pct_change,entry_wage
0,41-3031,"Securities, Commodities, and Financial Services Sales ...","4,145.00",-45.00,"4,190.00",-1.10,shrinking,-0.80,"97,955.00"
1,27-3041,Editors,"1,696.00",-114.00,"1,810.00",-6.70,shrinking,-6.10,"67,885.00"
2,53-2011,"Airline Pilots, Copilots, and Flight Engineers",753.00,-37.00,790.00,-4.90,shrinking,-5.10,"144,502.00"
3,41-4011,"Sales Representatives, Wholesale and Manufacturing, Te...",537.00,-23.00,560.00,-4.30,shrinking,-3.60,"79,914.00"
4,27-3023,"News Analysts, Reporters, and Journalists",505.00,-35.00,540.00,-6.90,shrinking,-6.70,"73,155.00"
5,27-4032,Film and Video Editors,403.00,-7.00,410.00,-1.70,shrinking,-1.40,"70,909.00"
6,13-2041,Credit Analysts,383.00,-27.00,410.00,-7.00,shrinking,-4.40,"96,156.00"
7,27-4012,Broadcast Technicians,338.00,-12.00,350.00,-3.60,shrinking,-3.00,"49,335.00"
8,15-1251,Computer Programmers,325.00,-55.00,380.00,-16.90,shrinking,-8.40,"91,963.00"
9,27-3011,Broadcast Announcers and Radio Disc Jockeys,177.00,-3.00,180.00,-1.70,shrinking,-1.60,"59,565.00"


### 7b. Industry churn from QWI, and each job's industry-weighted rate

In [16]:
qwi = q("""
select year, industry_key,
       sum(separations) as separations, sum(emp_begin) filter (where separations is not null) as emp_for_sep,
       sum(hires_new) as hires_new, sum(emp_begin) filter (where hires_new is not null) as emp_for_hire,
       count(*) as borough_quarters, count(separations) as sep_released
from fct_industry_hiring_quarterly
where breakdown = 'age' and group_code = 'A00' and year in (2019, 2024)
  and area_key in ('bronx', 'brooklyn', 'manhattan', 'queens', 'staten_island')
group by 1, 2
""")
qwi["sep_rate"] = qwi.separations / (qwi.emp_for_sep / 4)
qwi["hire_rate"] = qwi.hires_new / (qwi.emp_for_hire / 4)
ind_titles = q("select industry_key, industry_title from dim_industry").set_index("industry_key").industry_title
R = qwi.pivot(index="industry_key", columns="year", values=["sep_rate", "hire_rate", "separations", "hires_new"])
R.columns = [f"{a}_{b}" for a, b in R.columns]
R.insert(0, "industry", R.index.map(ind_titles).str.slice(0, 48))
R = R.sort_values("separations_2024", ascending=False)
show(R[["industry", "separations_2024", "hires_new_2024", "sep_rate_2024", "hire_rate_2024", "sep_rate_2019", "hire_rate_2019"]].round(3),
     "QWI NYC private: annual separation and new-hire rates by sector (Σ quarters ÷ mean beginning-of-quarter jobs), 2024 and 2019")
tot_row = qwi[(qwi.year == 2024) & (qwi.industry_key == "00")].iloc[0]
sec = qwi[(qwi.year == 2024) & (qwi.industry_key != "00")]
print(f"2024 check: Σ sector separations {sec.separations.sum():,.0f} vs the QWI total row {tot_row.separations:,.0f} "
      f"({sec.separations.sum() / tot_row.separations - 1:+.2%}); all-sector separation rate {tot_row.sep_rate:.1%}, new-hire rate {tot_row.hire_rate:.1%}")

**QWI NYC private: annual separation and new-hire rates by sector (Σ quarters ÷ mean beginning-of-quarter jobs), 2024 and 2019**

,industry,separations_2024,hires_new_2024,sep_rate_2024,hire_rate_2024,sep_rate_2019,hire_rate_2019
industry_key,,,,,,,
00,"Total, all industries","2,495,257.00","2,232,885.00",0.59,0.53,0.65,0.57
62,Health Care and Social Assistance,"481,129.00","458,684.00",0.49,0.46,0.47,0.45
72,Accommodation and Food Services,"293,210.00","269,206.00",0.84,0.77,0.94,0.86
56,Administrative and Support and Waste Management,"245,102.00","213,046.00",0.93,0.81,1.08,0.94
54,"Professional, Scientific, and Technical Services","218,274.00","180,687.00",0.49,0.41,0.55,0.49
51,Information,"212,606.00","147,773.00",0.84,0.58,1.12,0.82
48-49,Transportation and Warehousing,"175,548.00","241,110.00",0.87,1.20,0.62,0.61
44-45,Retail Trade,"172,518.00","155,562.00",0.58,0.52,0.69,0.62
52,Finance and Insurance,"123,803.00","101,474.00",0.33,0.27,0.34,0.29


2024 check: Σ sector separations 2,495,238 vs the QWI total row 2,495,257 (-0.00%); all-sector separation rate 59.2%, new-hire rate 52.9%


In [17]:
bij = q("select industry_key, soc_code, share_of_job from bridge_industry_job where share_of_job is not null")
rates = qwi[qwi.year == 2024].set_index("industry_key")[["sep_rate", "hire_rate"]]
bj = bij.merge(rates, left_on="industry_key", right_index=True, how="inner")      # drops GOV-OEWS, 92 and sectors without a QWI row
churn = (bj.assign(ws=bj.share_of_job * bj.sep_rate, wh=bj.share_of_job * bj.hire_rate)
           .groupby("soc_code").agg(coverage=("share_of_job", "sum"), ws=("ws", "sum"), wh=("wh", "sum"), sectors=("industry_key", "size")))
churn["industry sep rate"] = churn.ws / churn.coverage
churn["industry hire rate"] = churn.wh / churn.coverage
churn["low_coverage"] = churn.coverage < 0.70
S = S.merge(churn[["industry sep rate", "industry hire rate", "coverage", "low_coverage"]], left_on="soc_code", right_index=True, how="left")
print(f"degree-reachable leaves with an industry churn rate: {int(S['industry sep rate'].notna().sum())} of {len(S)}; coverage under 70% (flagged): {int(S.low_coverage.fillna(False).sum())}; "
      f"median coverage {S.coverage.median():.0%}")
cols7b = ["soc_code", "soc_title", "annual_openings_avg", "growth share %", "openings_from", "industry sep rate", "industry hire rate", "coverage", "low_coverage"]
list1b = S[S.soc_code.isin(top_open.soc_code)].set_index("soc_code").reindex(top_open.soc_code).reset_index()
list1b.index += 1
show(list1b[cols7b].round(3), "List 1 with the churn of each job's industries (QWI 2024, share_of_job-weighted)")
show(S[~S.is_all_other & ~S.low_coverage.fillna(True)].nlargest(10, "industry sep rate")[cols7b].round(3).reset_index(drop=True), "Degree-reachable jobs in the churniest industries")
show(S[~S.is_all_other & ~S.low_coverage.fillna(True)].nsmallest(10, "industry sep rate")[cols7b].round(3).reset_index(drop=True), "… and in the stickiest")
ok = S.dropna(subset=["industry sep rate"])
print(f"Spearman, growth share vs industry separation rate over {len(ok)} jobs: {ok['growth share %'].rank().corr(ok['industry sep rate'].rank()):.2f} "
      f"(expected weak: different things)")
sign_ok = int((np.sign(S.growth_share) == np.sign(S["pct_change"])).sum())
print(f"growth share and pct_change agree in sign on {sign_ok} of {len(S)} leaves")

degree-reachable leaves with an industry churn rate: 184 of 186; coverage under 70% (flagged): 16; median coverage 98%


**List 1 with the churn of each job's industries (QWI 2024, share_of_job-weighted)**

,soc_code,soc_title,annual_openings_avg,growth share %,openings_from,industry sep rate,industry hire rate,coverage,low_coverage
1,11-1021,General and Operations Managers,"9,781.00",9.52,mostly replacement,0.59,0.52,0.97,False
2,29-1141,Registered Nurses,"7,410.00",29.15,growth-led,0.52,0.48,0.96,False
3,13-2011,Accountants and Auditors,"6,648.00",9.15,mostly replacement,0.51,0.43,0.94,False
4,13-1161,Market Research Analysts and Marketing Specialists,"6,037.00",9.06,mostly replacement,0.56,0.47,1.00,False
5,13-1111,Management Analysts,"4,444.00",12.24,mixed,0.49,0.41,0.91,False
6,15-1252,Software Developers,"4,308.19",16.78,mixed,0.56,0.47,0.99,False
7,41-3031,"Securities, Commodities, and Financial Services Sales ...","4,145.00",-1.09,shrinking,0.34,0.28,1.00,False
8,25-2021,"Elementary School Teachers, Except Special Education","3,573.00",20.23,mixed,0.55,0.42,1.00,False
9,11-3031,Financial Managers,"3,334.00",19.02,mixed,0.44,0.37,0.97,False
10,13-1071,Human Resources Specialists,"3,099.00",14.17,mixed,0.58,0.51,0.91,False


**Degree-reachable jobs in the churniest industries**

,soc_code,soc_title,annual_openings_avg,growth share %,openings_from,industry sep rate,industry hire rate,coverage,low_coverage
0,13-1011,"Agents and Business Managers of Artists, Performers, a...",504.00,18.65,mixed,1.03,0.78,1.00,False
1,27-1013,"Fine Artists, Including Painters, Sculptors, and Illus...",263.00,12.55,mixed,0.99,0.74,1.00,False
2,25-4013,Museum Technicians and Conservators,130.00,15.38,mixed,0.98,0.74,0.89,False
3,11-9072,"Entertainment and Recreation Managers, Except Gambling",92.75,4.63,mostly replacement,0.91,0.71,0.75,False
4,53-2011,"Airline Pilots, Copilots, and Flight Engineers",753.00,-4.91,shrinking,0.87,1.20,0.99,False
5,27-4031,"Camera Operators, Television, Video, and Film",144.00,-4.17,shrinking,0.86,0.63,1.00,False
6,27-3011,Broadcast Announcers and Radio Disc Jockeys,177.00,-1.70,shrinking,0.84,0.59,1.00,False
7,27-1027,Set and Exhibit Designers,126.00,4.76,mostly replacement,0.83,0.71,1.00,False
8,27-4032,Film and Video Editors,403.00,-1.74,shrinking,0.82,0.60,1.00,False
9,27-3043,Writers and Authors,811.00,6.29,mostly replacement,0.81,0.61,1.00,False


**… and in the stickiest**

,soc_code,soc_title,annual_openings_avg,growth share %,openings_from,industry sep rate,industry hire rate,coverage,low_coverage
0,17-1021,Cartographers and Photogrammetrists,21.00,4.76,mostly replacement,0.32,0.29,0.67,True
1,13-2053,Insurance Underwriters,251.00,0.40,mostly replacement,0.33,0.27,0.97,False
2,41-3031,"Securities, Commodities, and Financial Services Sales ...","4,145.00",-1.09,shrinking,0.34,0.28,1.00,False
3,13-2052,Personal Financial Advisors,"1,348.00",8.75,mostly replacement,0.34,0.28,1.00,False
4,13-2061,Financial Examiners,"1,151.00",22.68,mixed,0.34,0.28,0.91,False
5,15-2011,Actuaries,130.00,30.77,growth-led,0.36,0.30,0.96,False
6,13-2072,Loan Officers,665.00,0.75,mostly replacement,0.36,0.30,0.99,False
7,13-2054,Financial Risk Specialists,641.54,7.39,mostly replacement,0.37,0.31,1.00,False
8,13-2041,Credit Analysts,383.00,-7.05,shrinking,0.40,0.34,1.00,False
9,41-4011,"Sales Representatives, Wholesale and Manufacturing, Te...",537.00,-4.28,shrinking,0.41,0.34,1.00,False


Spearman, growth share vs industry separation rate over 184 jobs: -0.08 (expected weak: different things)
growth share and pct_change agree in sign on 186 of 186 leaves


### 7c. Trace: Registered Nurses, industry churn by hand

In [18]:
rn_mix = bj[bj.soc_code == "29-1141"].sort_values("share_of_job", ascending=False)
rn_mix = rn_mix.assign(industry=rn_mix.industry_key.map(ind_titles).str.slice(0, 40), contribution=rn_mix.share_of_job * rn_mix.sep_rate)
show(rn_mix[["industry_key", "industry", "share_of_job", "sep_rate", "contribution"]].round(4), "RN: sectors with a readable share and a QWI rate")
rn_hand = rn_mix.contribution.sum() / rn_mix.share_of_job.sum()
rn_fact = S.loc[S.soc_code == "29-1141", "industry sep rate"].iloc[0]
print(f"by hand {rn_hand:.4f} vs table {rn_fact:.4f}; coverage {rn_mix.share_of_job.sum():.1%} (government hospitals sit in GOV-OEWS and drop out)")
assert abs(rn_hand - rn_fact) < 1e-9

**RN: sectors with a readable share and a QWI rate**

,industry_key,industry,share_of_job,sep_rate,contribution
2679,62,Health Care and Social Assistance,0.78,0.49,0.38
2145,56,Administrative and Support and Waste Man,0.08,0.93,0.08
2181,61,Educational Services,0.06,0.55,0.03
1294,52,Finance and Insurance,0.02,0.33,0.01
1735,55,Management of Companies and Enterprises,0.01,0.40,0.00
1608,54,"Professional, Scientific, and Technical",0.00,0.49,0.00
3170,81,Other Services (except Public Administra,0.00,0.52,0.00
272,31-33,Manufacturing,0.00,0.45,0.00
635,42,Wholesale Trade,0.00,0.38,0.00


by hand 0.5239 vs table 0.5239; coverage 95.9% (government hospitals sit in GOV-OEWS and drop out)

## 8. Findings (from the tables above; §7 added the same evening)

1. NYC added jobs since 2019, and all of the net gain is in degree-reachable work. QCEW NYC total 4.49 M → 4.65 M (+3.6%). Degree-reachable jobs 1.22 M → 1.45 M (+18.9%, +230 k); every other entry level together −2.1% (−69 k). Degree-reachable work went from 27.1% to 31.1% of NYC jobs. The published metro series agrees in direction (degree-reachable +9.9%, the rest −9.8%); its detailed-row total fell 4.4%, which is partly the 2024 metro redefinition and OEWS's three-year panel, so metro and NYC levels are not comparable, only directions.
2. **The gain is concentrated.** Management +55% (+117 k, half of the degree-reachable gain), Business and Financial Operations +19% (+62 k), Computer and Mathematical +25% (+32 k), Healthcare Practitioners +19% (+24 k). Shrinking: Arts, Design and Media −10%, Sales −17%, Architecture and Engineering −4%. Management's size is partly D8's doing: NYC's sector mix (finance, professional services, headquarters) is management-heavy on the state pattern, and the metro series shows the same managers growing 40–94%.
3. **List 1 (openings)** is led by General and Operations Managers (9.8 k a year), Registered Nurses (7.4 k), Accountants (6.6 k), Market Research Analysts (6.0 k), Management Analysts (4.4 k) and Software Developers (4.3 k, D1-imputed row). Twenty of the 25 are rated Favorable or Very Favorable by NYS DOL; the two Unfavorable ones (Securities Sales −0.8% projected, Editors −6.1%) are large but shrinking, which is exactly what S10 says to show, not hide. The 3 unrated rows are the D1 defect codes (Software Developers, Project Management Specialists, Financial Analysts).
4. List 2 (growth): tech is splitting in two, and management is where the growth is. Up: Software Developers +68% (41.6 k → 70.1 k), Software QA +66%, Information Security +46%, Data Scientists 1.1 k → 14.4 k (a new SOC 2018 code; the metro series grows 8.6× too, so most of it is reclassification plus real growth, flagged `split_2019`). Down, in the same metro: Computer Programmers −57%, Computer Network Support −49%, Systems Analysts −42%, Network Administrators −33%, Web Developers −40%. Management: Marketing Managers +158%, Project Management Specialists +108% (new code), Financial Managers +82%, Computer and IS Managers +74%, HR Managers +67%. Shrinking creative and sales jobs: Broadcast Technicians −64%, Advertising Managers −55%, Graphic Designers −36%, Fashion Designers −34%, Wholesale Sales Reps (technical) −35%.
5. **Checks.** NYC estimate vs metro published growth over the 132 floor jobs: r = 0.97, Spearman ρ = 0.83; 23 of 132 disagree on sign, mostly small jobs (Logisticians +101% vs −3%, Substitute Teachers +18% vs −12%). For `split_2019` rows the metro column is the combined code's growth (Web Developers and Web/Digital Interface Designers both show +42.6%), so read it as group growth there. NYS DOL's rating never says Very Unfavorable for a listed job; one List 2 job is Unfavorable (Sales Engineers: +27% actual, +4.8% projected). The 22 year-to-year swing cases all sit at or below 752 jobs, so the 1,000 floor removes every one of them while keeping 97% of degree-reachable employment. RN trace reproduces the fact to the unit (108,534); the 19 k gap to NYS DOL's NYC count is the known D8 over-prediction.
6. Limits to say on every page. The NYC job series is an estimate (QCEW × state pattern, D8/D18) and over-states jobs whose NYC staffing differs from the state's (RNs, home health); the metro is twice NYC; 2019 is a hybrid-code year; 2022-32 rows flagged `imputed` are D1 fills; QCEW excludes the self-employed (writers, designers, consultants are under-counted); openings are a ten-year average.
7. **For the mart.** Keep both S10 lists as built here (floor 1,000, All Other to its own list, split flag), show the NYS DOL 2025 count and the metro series beside the estimate, and consider a third column, absolute change, for the growth page (Marketing Managers +19 k and Software Developers +28 k are the headline numbers; Data Scientists' +1,217% is not). Decided (plan §3b item 4): the growth list ranks by absolute change with percent beside it; the percent ranking stays as a toggle.
8. Why the openings exist (§7, plan §3g): NYC's degree-reachable openings are 87% replacement. Of 136,787 openings a year on the 186 leaves, 17,894 (13.1%) come from growth and 118,893 from people leaving (retirements, career changes; modeled by the BLS separations method), against 9.7% growth for every NYC leaf. Bands: growth-led 15 jobs, 9.6% of openings, and it is a health-care list plus two tech jobs (Registered Nurses 29% growth share, Medical and Health Services Managers 36%, Data Scientists 33%, Information Security Analysts 31%, Clinical Lab Technologists, Radiologic Technologists, Sonographers, Respiratory Therapists, Actuaries); mixed 110 jobs, 47.5% (Software Developers 17%, teachers 20–22%, Financial Managers 19%, Management Analysts 12%); mostly replacement 47 jobs, 35.9% (General and Operations Managers 9.5%, Accountants 9.1%, Market Research Analysts 9.1%, Marketing Managers 8.5%, Financial Analysts 7.9%, Paralegals 5.2%, Producers and Directors 2.0%); shrinking 14 jobs, 7.0%, which still open because people leave faster than the job shrinks (Securities Sales Agents 4,145 openings a year with −45 growth, Editors −6.7%, Airline Pilots, Computer Programmers −17%, Credit Analysts −7%). So List 1 is mostly a list of jobs that hire because people leave; among its top six only Registered Nurses grow. Industry churn (QWI 2024, five boroughs, private, all ages): NYC separations run at 59% of beginning-of-quarter jobs a year and new hires at 53% (2019: 65% / 57%; both lower now). Sticky sectors: Finance 33%, Real Estate 36%, Wholesale 38%, Management of Companies 40%, Manufacturing 45%; churny: Arts and Recreation 105%, Administrative and Support 93%, Transportation 87% (the 2024 Q1 employer reclassification, nb 41: 2019 was 62% and the 2024 new-hire rate is 120%, so footnote it), Accommodation and Food 84%, Information 84%; Health Care 49%, Professional and Technical 49%, Education 55%. Weighted by each job's sector mix (`share_of_job`, NY May 2025), List 1 jobs sit in industries that separate 41–60% of jobs a year (Financial Analysts 41%, Financial Managers 44%, Management Analysts 49%, RNs 52%, General Managers 59%, Project Management Specialists 60%); the arts jobs sit at 80–103% (Agents of Artists 103%, Fine Artists 99%, Writers and Authors 81%), the finance jobs at 33–37%. Growth share and industry churn are unrelated (Spearman −0.08 over 184 jobs), which is the point: the two columns say different things. Checks: openings = growth + separations on all 592 leaves (0 off by more than 1); Σ sector separations vs the QWI total row −0.00%; RN churn by hand 0.5239 = table (78% of RNs in sector 62, coverage 96%, government hospitals drop out); growth share and `pct_change` agree in sign on 186 of 186; 184 of 186 leaves get a churn rate, 16 flagged under 70% coverage (public-sector jobs: QWI has no government), median coverage 98%. Limits: separations are a projection-model quantity, not observed turnover; QWI is industry churn (every occupation in the sector, private only, not seasonally adjusted) and NYC runs +3–6% above QCEW in level; Transportation 2024 is a break. For the mart: List 1 carries the growth share with its band as the label. Recommendation: the QWI rate belongs on the industry pages as a sector number; on a job page, at most one line ("the industries this job sits in turn over about X% of their jobs a year"), never as a ranking column, because a job-weighted industry rate is not the job's own turnover.